# TACS - Task-Aligned Context Selection - Stanford Cars

Notebook này triển khai method cuối trong Table 2 của paper: **TACS (Ours)**.

Bản chất TACS: thay vì chọn context theo random hoặc similarity cố định, mô hình học một **Selector** để chọn ảnh context thật sự giúp downstream classifier giảm lỗi. Selector tính utility score giữa query và từng candidate, biến score thành xác suất chọn context, rồi được train bằng hai tín hiệu cùng lúc:

- **Differentiable path**: dùng straight-through Gumbel-Softmax để chọn một context rời rạc ở forward pass nhưng vẫn cho gradient chảy về Selector qua task loss `L_grad = CE(f_d(x_q, x_sel), y)`.
- **Policy path**: sample một context theo policy của Selector, đo reward theo mức cải thiện downstream loss `reward = CE(f_d(x_q, empty), y) - CE(f_d(x_q, x_selected), y)`, rồi cập nhật Selector bằng policy gradient.
- **Joint objective**: `L_TACS = L_grad + lambda * L_policy`.

Notebook vẫn giữ standard train/test split của `archive (13)`, không tạo validation split, lưu checkpoint sau từng epoch, và có cell in `test_accuracy` để so sánh với các method trước.

Lưu ý thực dụng: paper dùng fixed candidate pool 20% train. Việc score toàn bộ pool trainable ở mọi batch rất nặng, nên notebook mặc định train trên mini-candidate set lấy từ fixed pool bằng `SELECTOR_CANDIDATES_PER_QUERY`. Khi test, Selector score **toàn bộ candidate pool** để chọn context bằng `argmax`, đúng tinh thần inference của TACS. Nếu GPU đủ mạnh, có thể tăng `SELECTOR_CANDIDATES_PER_QUERY` để gần paper hơn.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -q /content/drive/MyDrive/archive\ \(13\).zip -d /content/archive\ \(13\)/

## 1. Cài thư viện

Chạy cell này trên Colab nếu môi trường chưa có đủ thư viện. Nếu chạy local đã cài sẵn thì có thể bỏ qua.


In [ ]:
%pip -q install torch torchvision pandas pillow tqdm matplotlib torchmetrics


In [ ]:
!wget -c 'https://dinov3.llamameta.net/dinov3_vits16/dinov3_vits16_pretrain_lvd1689m-08c60483.pth?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoiNTFma3Jxa3l3bTl3MGpweHI2NndvNWZsIiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXRcLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3ODkyOTc0MTJ9fX1dfQ__&Signature=sdowk%7EiiF6j8599sB9K5-M3sB%7Elq1be-oKP%7El7VBISEmjHEBPyCWw9Y2JD4oJY9LbXE313Q4ZgjX%7EXO3EiwGQZ3MZhfkZr0OXmADX8L%7E7XX%7E4mkVA1JSOkVzEa3Oj39gY%7E%7ER21%7EmJXEcFOZPMj%7EzGs-uxIh1xeRrfo-ZUqv3Yz7z7vUgn1OoDhu98kPnh8KT4MF1Gd6gcB4LogqWk3ooVScR6z8ECw50eawfY8lyUUzv%7E%7Em1QdstSU1k6HFhKFbuuogf0MyH7rzYScskxtGG95ke47CKn6CWSNbHLKhcLDfNx-XrN0A-xxis10rpJaMh2jBmxwohSr8XQlGVCBPWqQ__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=1749739516259745' \
  -O /content/dinov3_vits16_pretrain_lvd1689m-08c60483.pth

## 2. Import, cấu hình, dataset và candidate pool

Cell này đọc standard split từ `archive (13)` gồm `anno_train.csv`, `anno_test.csv`, `names.csv` và ảnh trong `car_data/car_data/train|test`. Candidate pool lấy cố định 20% train split. Không tách validation vì benchmark đang so sánh theo standard train/test split.


In [ ]:
import json
import random
import subprocess
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

PROJECT_DIR = Path.cwd()
DATASET_ROOT_CANDIDATES = [
    PROJECT_DIR / 'archive (13)',
    Path('F:/Documents/CODE/Python/cv_project/XLA/archive (13)'),
    Path('/content/archive (13)'),
    Path('/content/drive/MyDrive/archive (13)'),
]
DATASET_ROOT = next(
    (p for p in DATASET_ROOT_CANDIDATES if (p / 'anno_train.csv').is_file() and (p / 'anno_test.csv').is_file() and (p / 'names.csv').is_file()),
    DATASET_ROOT_CANDIDATES[0],
)
TRAIN_IMAGE_ROOT = DATASET_ROOT / 'car_data' / 'car_data' / 'train'
TEST_IMAGE_ROOT = DATASET_ROOT / 'car_data' / 'car_data' / 'test'

OUT_PUT_DIR = PROJECT_DIR / 'outputs' / 'baseline_tacs_cars'
OUTPUT_DIR = OUT_PUT_DIR
OUT_PUT_DIR.mkdir(parents=True, exist_ok=True)

DINOV3_REPO_URL = 'https://github.com/facebookresearch/dinov3.git'
DINOV3_REPO_DIR = str(PROJECT_DIR / 'dinov3')
DINOV3_MODEL_NAME = 'dinov3_vits16'
DINOV3_WEIGHTS_URL = 'https://dl.fbaipublicfiles.com/dinov3/dinov3_vits16/dinov3_vits16_pretrain_lvd1689m-08c60483.pth'
DINOV3_WEIGHTS = str(PROJECT_DIR / 'dinov3_vits16_pretrain_lvd1689m-08c60483.pth')
BACKBONE_NAME = DINOV3_MODEL_NAME
FEATURE_DIM = 384

IMAGE_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 100
TASK_LEARNING_RATE = 1e-4
SELECTOR_LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
NUM_CLASSES = 196
TOP_K = 5
USE_BBOX_CROP = False
CANDIDATE_POOL_FRACTION = 0.20
SELECTOR_CANDIDATES_PER_QUERY = 16
GUMBEL_TEMPERATURE = 0.1
HYBRID_LAMBDA = 1.0
POLICY_ENTROPY_WEIGHT = 0.0
GRAD_CLIP_NORM = 1.0
SELECTOR_NORMALIZE_FEATURES = True
SELECTOR_EMBED_BATCH_SIZE = 64
EVALUATION_QUERY_BATCH_SIZE = 16

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomResizedCrop((IMAGE_SIZE, IMAGE_SIZE), scale=(0.75, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])


def load_class_names(dataset_root):
    class_names = pd.read_csv(dataset_root / 'names.csv', header=None)[0].astype(str).tolist()
    if len(class_names) != NUM_CLASSES:
        raise ValueError(f'Expected {NUM_CLASSES} classes, got {len(class_names)}')
    return class_names


def load_annotations(dataset_root, split):
    df = pd.read_csv(
        dataset_root / f'anno_{split}.csv',
        header=None,
        names=['fname', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 'class_id_1based'],
    )
    df['class_id'] = df['class_id_1based'].astype(int) - 1
    df['split'] = split
    df['row_id'] = np.arange(len(df))
    return df


def attach_image_paths(df, image_root):
    image_index = {path.name: path for path in sorted(Path(image_root).rglob('*.jpg'))}
    missing = [fname for fname in df['fname'] if fname not in image_index]
    if missing:
        raise FileNotFoundError(f'Thieu anh trong {image_root}, vi du: {missing[:5]}')
    out = df.copy()
    out['image_path'] = [str(image_index[fname]) for fname in out['fname']]
    return out


class_names = load_class_names(DATASET_ROOT)
train_df = attach_image_paths(load_annotations(DATASET_ROOT, 'train'), TRAIN_IMAGE_ROOT).reset_index(drop=True)
test_df = attach_image_paths(load_annotations(DATASET_ROOT, 'test'), TEST_IMAGE_ROOT).reset_index(drop=True)
NUM_CLASSES = len(class_names)

candidate_rng = np.random.default_rng(SEED)
candidate_pool_size = max(1, int(round(len(train_df) * CANDIDATE_POOL_FRACTION)))
candidate_pool_positions = np.sort(candidate_rng.choice(len(train_df), size=candidate_pool_size, replace=False))
candidate_df = train_df.iloc[candidate_pool_positions].reset_index(drop=True)
candidate_df.to_csv(OUT_PUT_DIR / 'candidate_pool.csv', index=False)

print('DATASET_ROOT:', DATASET_ROOT.resolve())
print('TRAIN_IMAGE_ROOT:', TRAIN_IMAGE_ROOT.resolve())
print('TEST_IMAGE_ROOT:', TEST_IMAGE_ROOT.resolve())
print('OUT_PUT_DIR:', OUT_PUT_DIR.resolve())
print(f'Standard train images: {len(train_df):,}')
print(f'Standard test images: {len(test_df):,}')
print(f'Candidate pool: {len(candidate_df):,}/{len(train_df):,} ({CANDIDATE_POOL_FRACTION:.0%})')
print(f'Train mini-candidates/query: {SELECTOR_CANDIDATES_PER_QUERY}')
print(f'Gumbel temperature: {GUMBEL_TEMPERATURE}')
print(f'Hybrid lambda: {HYBRID_LAMBDA}')
print(f'Train label range: {int(train_df.class_id.min())}..{int(train_df.class_id.max())}')
print(f'Test label range: {int(test_df.class_id.min())}..{int(test_df.class_id.max())}')
display(train_df.head())
display(candidate_df.head())


## 3. Dataset cho TACS

Training sample gồm query image và một mini-candidate set lấy từ fixed candidate pool. Selector sẽ chọn context trong mini-set này bằng Gumbel-Softmax/policy sampling. Test sample chỉ chứa query; ở cell evaluation, Selector đã train sẽ score toàn bộ candidate pool để chọn context bằng `argmax`.


In [ ]:
class SimpleImageDataset(Dataset):
    def __init__(self, annotations, transform, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.transform = transform
        self.crop_bbox = crop_bbox

    def __len__(self):
        return len(self.annotations)

    def _open_image(self, row):
        image = Image.open(row['image_path']).convert('RGB')
        if self.crop_bbox:
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = int(row['bbox_x2'])
            bottom = int(row['bbox_y2'])
            image = image.crop((left, top, right, bottom))
        return image

    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        image = self.transform(self._open_image(row))
        return image, int(index), row['fname'], int(row['class_id'])


class StanfordCarsTACSTrainDataset(Dataset):
    def __init__(self, annotations, candidate_annotations, query_transform, candidate_transform, candidates_per_query=16, seed=42, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.candidate_annotations = candidate_annotations.reset_index(drop=True).copy()
        self.query_transform = query_transform
        self.candidate_transform = candidate_transform
        self.candidates_per_query = int(candidates_per_query)
        self.seed = int(seed)
        self.epoch = 0
        self.crop_bbox = crop_bbox
        self.candidate_row_ids = self.candidate_annotations['row_id'].astype(int).to_numpy()
        if self.candidates_per_query < 1:
            raise ValueError('candidates_per_query phai >= 1')

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.annotations)

    def _open_image(self, row):
        image = Image.open(row['image_path']).convert('RGB')
        if self.crop_bbox:
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = int(row['bbox_x2'])
            bottom = int(row['bbox_y2'])
            image = image.crop((left, top, right, bottom))
        return image

    def _sample_candidate_positions(self, index):
        query_row_id = int(self.annotations.iloc[index]['row_id'])
        valid_positions = np.flatnonzero(self.candidate_row_ids != query_row_id)
        if len(valid_positions) == 0:
            valid_positions = np.arange(len(self.candidate_annotations))
        rng = np.random.default_rng(self.seed + self.epoch * 1_000_003 + index)
        replace = len(valid_positions) < self.candidates_per_query
        return rng.choice(valid_positions, size=self.candidates_per_query, replace=replace)

    def __getitem__(self, index):
        query_row = self.annotations.iloc[index]
        candidate_positions = self._sample_candidate_positions(index)
        candidate_rows = self.candidate_annotations.iloc[candidate_positions]
        query_image = self.query_transform(self._open_image(query_row))
        candidate_images = [self.candidate_transform(self._open_image(row)) for _, row in candidate_rows.iterrows()]
        candidate_images = torch.stack(candidate_images, dim=0)
        candidate_labels = torch.as_tensor(candidate_rows['class_id'].astype(int).to_numpy(), dtype=torch.long)
        candidate_positions_tensor = torch.as_tensor(candidate_positions.astype(np.int64), dtype=torch.long)
        return query_image, candidate_images, int(query_row['class_id']), query_row['fname'], candidate_positions_tensor, candidate_labels


train_dataset = StanfordCarsTACSTrainDataset(
    train_df,
    candidate_df,
    train_transform,
    train_transform,
    candidates_per_query=SELECTOR_CANDIDATES_PER_QUERY,
    seed=SEED,
    crop_bbox=USE_BBOX_CROP,
)
test_dataset = SimpleImageDataset(test_df, eval_transform, crop_bbox=USE_BBOX_CROP)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=EVALUATION_QUERY_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

sample = train_dataset[0]
print('Query tensor:', sample[0].shape)
print('Candidate tensor:', sample[1].shape)
print('Query fname:', sample[3])
print('Candidate labels sample:', sample[5][:8].tolist())


## 4. DINOv3 Selector và Downstream Task Network

TACS có hai module train chung:

- `TACSSelector`: DINOv3 ViT-S/16 nhận query/candidate, lấy CLS embedding, tính utility score `s_i = z_q^T z_i`.
- `DinoV3ContextClassifier`: downstream classifier nhận `(query_image, selected_context_image)`. Nếu DINOv3 expose token API, model sẽ concat patch tokens của query và context trước transformer blocks; nếu không, notebook fallback sang feature fusion để vẫn chạy được.


In [ ]:
def ensure_dinov3_assets():
    repo_dir = Path(DINOV3_REPO_DIR)
    weights_path = Path(DINOV3_WEIGHTS)
    if not repo_dir.is_dir():
        try:
            print('Cloning DINOv3 repo to:', repo_dir)
            subprocess.run(['git', 'clone', '--depth', '1', DINOV3_REPO_URL, str(repo_dir)], check=True)
        except Exception as exc:
            print('Khong clone duoc DINOv3 repo. torch.hub se thu tai tu GitHub khi tao model.')
            print(type(exc).__name__ + ':', exc)
    else:
        print('DINOv3 repo da ton tai:', repo_dir)
    if not weights_path.is_file():
        print('Dang tai DINOv3 ViT-S/16 weights ve:', weights_path)
        urlretrieve(DINOV3_WEIGHTS_URL, weights_path)
    else:
        print('DINOv3 weights da ton tai:', weights_path)
    if weights_path.stat().st_size < 10 * 1024 * 1024:
        raise ValueError(f'File weight co ve bi loi hoac qua nho: {weights_path}')
    print('Weights size MB:', round(weights_path.stat().st_size / 1024**2, 2))


def resolve_dinov3_weights():
    path = Path(DINOV3_WEIGHTS)
    if not path.is_file():
        raise FileNotFoundError(f'Khong tim thay file: {path}')
    if path.stat().st_size < 10 * 1024 * 1024:
        raise ValueError('File weight bi rong hoac bi hong')
    return str(path)


def load_dinov3_model():
    repo_dir = Path(DINOV3_REPO_DIR)
    source = 'local' if repo_dir.is_dir() else 'github'
    repo_or_dir = str(repo_dir) if repo_dir.is_dir() else 'facebookresearch/dinov3'
    return torch.hub.load(repo_or_dir, DINOV3_MODEL_NAME, source=source, weights=resolve_dinov3_weights())


def extract_cls_features(features):
    if isinstance(features, dict):
        if 'x_norm_clstoken' in features:
            return features['x_norm_clstoken']
        if 'x_norm_cls_token' in features:
            return features['x_norm_cls_token']
        raise KeyError(f'Khong tim thay CLS token trong output keys: {list(features)}')
    return features


class TACSSelector(nn.Module):
    def __init__(self, normalize_features=True):
        super().__init__()
        self.backbone = load_dinov3_model()
        self.normalize_features = bool(normalize_features)
        self.feature_dim = FEATURE_DIM

    def encode_images(self, images):
        features = extract_cls_features(self.backbone(images)).float()
        if self.normalize_features:
            features = F.normalize(features, dim=1)
        return features

    def score_candidates(self, query_images, candidate_images):
        batch_size, num_candidates = candidate_images.shape[:2]
        query_features = self.encode_images(query_images)
        flat_candidates = candidate_images.reshape(batch_size * num_candidates, *candidate_images.shape[2:])
        candidate_features = self.encode_images(flat_candidates).reshape(batch_size, num_candidates, -1)
        return torch.einsum('bd,bmd->bm', query_features, candidate_features)


class DinoV3ContextClassifier(nn.Module):
    def __init__(self, num_classes, feature_dim=384, allow_feature_fallback=True):
        super().__init__()
        self.backbone = load_dinov3_model()
        self.feature_dim = feature_dim
        self.allow_feature_fallback = allow_feature_fallback
        self.supports_token_fusion = all(hasattr(self.backbone, attr) for attr in ['blocks', 'norm']) and (hasattr(self.backbone, 'prepare_tokens_with_masks') or hasattr(self.backbone, 'prepare_tokens'))
        self.active_fusion_mode = 'token_concat' if self.supports_token_fusion else 'feature_fusion'
        self.token_head = nn.Linear(feature_dim, num_classes)
        self.feature_head = nn.Sequential(nn.LayerNorm(feature_dim * 4), nn.Linear(feature_dim * 4, num_classes))

    def _encode_single(self, images):
        return extract_cls_features(self.backbone(images))

    def _prepare_tokens(self, images):
        if hasattr(self.backbone, 'prepare_tokens_with_masks'):
            try:
                return self.backbone.prepare_tokens_with_masks(images, None)
            except TypeError:
                return self.backbone.prepare_tokens_with_masks(images)
        if hasattr(self.backbone, 'prepare_tokens'):
            return self.backbone.prepare_tokens(images)
        raise AttributeError('Backbone khong expose prepare_tokens API')

    def _forward_token_concat(self, query_images, context_images):
        query_tokens = self._prepare_tokens(query_images)
        context_tokens = self._prepare_tokens(context_images)
        num_register_tokens = int(getattr(self.backbone, 'num_register_tokens', 0))
        num_prefix_tokens = 1 + num_register_tokens
        pair_tokens = torch.cat([query_tokens[:, :num_prefix_tokens], query_tokens[:, num_prefix_tokens:], context_tokens[:, num_prefix_tokens:]], dim=1)
        for block in self.backbone.blocks:
            pair_tokens = block(pair_tokens)
        pair_tokens = self.backbone.norm(pair_tokens)
        return self.token_head(pair_tokens[:, 0])

    def _forward_feature_fusion(self, query_images, context_images):
        query_features = self._encode_single(query_images)
        context_features = self._encode_single(context_images)
        pair_features = torch.cat([query_features, context_features, torch.abs(query_features - context_features), query_features * context_features], dim=1)
        return self.feature_head(pair_features)

    def forward(self, query_images, context_images):
        if self.supports_token_fusion:
            try:
                self.active_fusion_mode = 'token_concat'
                return self._forward_token_concat(query_images, context_images)
            except (AttributeError, TypeError, KeyError) as exc:
                if not self.allow_feature_fallback:
                    raise
                self.supports_token_fusion = False
                self.active_fusion_mode = 'feature_fusion'
                print('Token-level fusion khong kha dung, fallback sang feature fusion:', type(exc).__name__, exc)
        self.active_fusion_mode = 'feature_fusion'
        return self._forward_feature_fusion(query_images, context_images)

    def forward_no_context(self, query_images):
        null_context = torch.zeros_like(query_images)
        return self.forward(query_images, null_context)


def straight_through_gumbel_context(selector_logits, candidate_images):
    selection_weights = F.gumbel_softmax(selector_logits, tau=GUMBEL_TEMPERATURE, hard=True, dim=1)
    selected_context = torch.einsum('bm,bmchw->bchw', selection_weights.to(candidate_images.dtype), candidate_images)
    return selected_context, selection_weights


ensure_dinov3_assets()
selector = TACSSelector(normalize_features=SELECTOR_NORMALIZE_FEATURES).to(DEVICE)
task_model = DinoV3ContextClassifier(NUM_CLASSES, FEATURE_DIM).to(DEVICE)
criterion = nn.CrossEntropyLoss()
criterion_per_sample = nn.CrossEntropyLoss(reduction='none')
optimizer = torch.optim.AdamW(
    [
        {'params': selector.parameters(), 'lr': SELECTOR_LEARNING_RATE},
        {'params': task_model.parameters(), 'lr': TASK_LEARNING_RATE},
    ],
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')
checkpoint_path = OUT_PUT_DIR / 'final_tacs_dinov3_vits16_cars.pt'
last_checkpoint_path = OUT_PUT_DIR / 'checkpoint_last.pt'
print('Initial downstream fusion mode:', task_model.active_fusion_mode)
print('Selector normalize features:', SELECTOR_NORMALIZE_FEATURES)
print('Checkpoint path:', checkpoint_path.resolve())


## 5. Train TACS và lưu checkpoint sau từng epoch

Mỗi batch train thực hiện đúng hai nhánh:

1. `L_grad`: Selector score mini-candidate set, chọn context bằng straight-through Gumbel-Softmax, downstream classifier học từ context đó.
2. `L_policy`: Selector sample action theo categorical policy, reward bằng độ giảm loss khi dùng context so với no-context/null context, advantage được chuẩn hóa trong batch.

Sau mỗi epoch notebook lưu `checkpoint_last.pt` ngay vào `OUT_PUT_DIR` để resume nếu Colab dừng giữa chừng.


In [ ]:
def standardize_advantage(rewards):
    rewards = rewards.float()
    if rewards.numel() <= 1:
        return rewards - rewards.mean()
    return (rewards - rewards.mean()) / (rewards.std(unbiased=False) + 1e-6)


def train_one_epoch(selector_model, task_network, loader, epoch):
    selector_model.train()
    task_network.train()
    loader.dataset.set_epoch(epoch)
    total_loss = 0.0
    total_grad_loss = 0.0
    total_policy_loss = 0.0
    total_entropy = 0.0
    total_reward = 0.0
    total_reward_sq = 0.0
    total_grad_selected_same_class = 0
    total_policy_selected_same_class = 0
    total_correct = 0
    total_examples = 0
    progress = tqdm(loader, desc=f'train epoch {epoch}', leave=False)

    for query_images, candidate_images, labels, _, candidate_positions, candidate_labels in progress:
        query_images = query_images.to(DEVICE, non_blocking=True)
        candidate_images = candidate_images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        candidate_positions = candidate_positions.to(DEVICE, non_blocking=True)
        candidate_labels = candidate_labels.to(DEVICE, non_blocking=True)
        batch_size = query_images.size(0)
        batch_indices = torch.arange(batch_size, device=DEVICE)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            selector_logits = selector_model.score_candidates(query_images, candidate_images)
            grad_context_images, grad_selection_weights = straight_through_gumbel_context(selector_logits, candidate_images)
            grad_logits = task_network(query_images, grad_context_images)
            grad_loss = criterion(grad_logits, labels)

        policy_dist = torch.distributions.Categorical(logits=selector_logits.float())
        policy_actions = policy_dist.sample()
        log_probs = policy_dist.log_prob(policy_actions)
        entropy = policy_dist.entropy().mean()
        policy_context_images = candidate_images[batch_indices, policy_actions]

        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                no_context_logits = task_network.forward_no_context(query_images)
                no_context_losses = criterion_per_sample(no_context_logits, labels)
                policy_context_logits = task_network(query_images, policy_context_images)
                policy_context_losses = criterion_per_sample(policy_context_logits, labels)
            rewards = no_context_losses.float() - policy_context_losses.float()
            advantages = standardize_advantage(rewards)

        policy_loss = -(log_probs * advantages.detach()).mean()
        loss = grad_loss + HYBRID_LAMBDA * policy_loss - POLICY_ENTROPY_WEIGHT * entropy
        scaler.scale(loss).backward()
        if GRAD_CLIP_NORM is not None and GRAD_CLIP_NORM > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(list(selector_model.parameters()) + list(task_network.parameters()), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        predictions = grad_logits.argmax(dim=1)
        grad_actions = grad_selection_weights.detach().argmax(dim=1)
        grad_selected_labels = candidate_labels[batch_indices, grad_actions]
        policy_selected_labels = candidate_labels[batch_indices, policy_actions]

        total_examples += batch_size
        total_loss += float(loss.detach().cpu()) * batch_size
        total_grad_loss += float(grad_loss.detach().cpu()) * batch_size
        total_policy_loss += float(policy_loss.detach().cpu()) * batch_size
        total_entropy += float(entropy.detach().cpu()) * batch_size
        total_reward += float(rewards.detach().sum().cpu())
        total_reward_sq += float((rewards.detach() ** 2).sum().cpu())
        total_correct += int((predictions == labels).detach().sum().cpu())
        total_grad_selected_same_class += int((grad_selected_labels == labels).detach().sum().cpu())
        total_policy_selected_same_class += int((policy_selected_labels == labels).detach().sum().cpu())

        progress.set_postfix(
            loss=f'{float(loss.detach().cpu()):.4f}',
            grad=f'{float(grad_loss.detach().cpu()):.4f}',
            policy=f'{float(policy_loss.detach().cpu()):.4f}',
            reward=f'{float(rewards.mean().detach().cpu()):.4f}',
        )

    reward_mean = total_reward / max(1, total_examples)
    reward_var = max(0.0, total_reward_sq / max(1, total_examples) - reward_mean ** 2)
    return {
        'loss': total_loss / max(1, total_examples),
        'grad_loss': total_grad_loss / max(1, total_examples),
        'policy_loss': total_policy_loss / max(1, total_examples),
        'policy_entropy': total_entropy / max(1, total_examples),
        'reward_mean': reward_mean,
        'reward_std': reward_var ** 0.5,
        'train_accuracy': total_correct / max(1, total_examples),
        'grad_selected_same_class_rate': total_grad_selected_same_class / max(1, total_examples),
        'policy_selected_same_class_rate': total_policy_selected_same_class / max(1, total_examples),
    }


def build_checkpoint_payload(epoch, history):
    return {
        'epoch': int(epoch),
        'method': 'TACS',
        'selector_state_dict': selector.state_dict(),
        'task_model_state_dict': task_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'history': list(history),
        'class_names': class_names,
        'architecture': BACKBONE_NAME,
        'feature_dim': FEATURE_DIM,
        'fusion_mode': getattr(task_model, 'active_fusion_mode', 'unknown'),
        'dinov3_model_name': DINOV3_MODEL_NAME,
        'dinov3_repo_dir': str(DINOV3_REPO_DIR),
        'dinov3_weights': str(resolve_dinov3_weights()),
        'dataset': 'Stanford Cars',
        'dataset_root': str(DATASET_ROOT),
        'standard_train_size': len(train_dataset),
        'standard_test_size': len(test_dataset),
        'image_size': IMAGE_SIZE,
        'mean': imagenet_mean,
        'std': imagenet_std,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'seed': SEED,
        'use_bbox_crop': USE_BBOX_CROP,
        'candidate_pool_fraction': CANDIDATE_POOL_FRACTION,
        'candidate_pool_size': len(candidate_df),
        'candidate_pool_positions': [int(x) for x in candidate_pool_positions],
        'selector_candidates_per_query': SELECTOR_CANDIDATES_PER_QUERY,
        'gumbel_temperature': GUMBEL_TEMPERATURE,
        'hybrid_lambda': HYBRID_LAMBDA,
        'policy_entropy_weight': POLICY_ENTROPY_WEIGHT,
        'selector_normalize_features': SELECTOR_NORMALIZE_FEATURES,
        'retriever': 'learned TACS selector with Gumbel-Softmax and policy-gradient reward',
        'pretraining': 'DINOv3',
    }


RESUME_FROM_LAST_CHECKPOINT = False
history = []
start_epoch = 1
OUT_PUT_DIR.mkdir(parents=True, exist_ok=True)

if RESUME_FROM_LAST_CHECKPOINT and last_checkpoint_path.is_file():
    saved_checkpoint = torch.load(last_checkpoint_path, map_location=DEVICE)
    selector.load_state_dict(saved_checkpoint['selector_state_dict'])
    task_model.load_state_dict(saved_checkpoint['task_model_state_dict'])
    if 'optimizer_state_dict' in saved_checkpoint:
        optimizer.load_state_dict(saved_checkpoint['optimizer_state_dict'])
        for state in optimizer.state.values():
            for key, value in state.items():
                if torch.is_tensor(value):
                    state[key] = value.to(DEVICE)
    if 'scheduler_state_dict' in saved_checkpoint:
        scheduler.load_state_dict(saved_checkpoint['scheduler_state_dict'])
    if 'scaler_state_dict' in saved_checkpoint:
        scaler.load_state_dict(saved_checkpoint['scaler_state_dict'])
    history = saved_checkpoint.get('history', [])
    start_epoch = int(saved_checkpoint.get('epoch', 0)) + 1
    print(f'Resumed from epoch {start_epoch - 1}: {last_checkpoint_path.resolve()}')

for epoch in range(start_epoch, EPOCHS + 1):
    train_metrics = train_one_epoch(selector, task_model, train_loader, epoch)
    scheduler.step()
    history.append({
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_grad_loss': train_metrics['grad_loss'],
        'train_policy_loss': train_metrics['policy_loss'],
        'policy_entropy': train_metrics['policy_entropy'],
        'reward_mean': train_metrics['reward_mean'],
        'reward_std': train_metrics['reward_std'],
        'train_accuracy': train_metrics['train_accuracy'],
        'grad_selected_same_class_rate': train_metrics['grad_selected_same_class_rate'],
        'policy_selected_same_class_rate': train_metrics['policy_selected_same_class_rate'],
        'task_learning_rate': scheduler.get_last_lr()[1],
        'selector_learning_rate': scheduler.get_last_lr()[0],
        'fusion_mode': getattr(task_model, 'active_fusion_mode', 'unknown'),
    })
    history_df = pd.DataFrame(history)
    history_df.to_csv(OUT_PUT_DIR / 'training_history.csv', index=False)
    torch.save(build_checkpoint_payload(epoch, history), last_checkpoint_path)
    print(
        f"Epoch {epoch:03d}/{EPOCHS} | loss {train_metrics['loss']:.4f} | "
        f"grad {train_metrics['grad_loss']:.4f} | policy {train_metrics['policy_loss']:.4f} | "
        f"reward {train_metrics['reward_mean']:.4f} | train acc {train_metrics['train_accuracy']:.4f}"
    )
    print(f'Saved latest checkpoint: {last_checkpoint_path.resolve()}')

final_epoch = history[-1]['epoch'] if history else 0
torch.save(build_checkpoint_payload(final_epoch, history), checkpoint_path)
history_df = pd.DataFrame(history)
history_df.to_csv(OUT_PUT_DIR / 'training_history.csv', index=False)
print('Saved final checkpoint:', checkpoint_path.resolve())
print('Saved latest checkpoint:', last_checkpoint_path.resolve())


## 6. Vẽ training history

Cell này đọc `training_history.csv` và vẽ các đại lượng quan trọng của TACS: train accuracy, task loss, policy loss, reward, entropy và tỷ lệ context cùng class. Reward dương nghĩa là context được sample giúp giảm loss so với no-context/null context.


In [ ]:
if 'history_df' not in globals():
    history_path = OUT_PUT_DIR / 'training_history.csv'
    if history_path.is_file():
        history_df = pd.read_csv(history_path)
    else:
        history_df = pd.DataFrame()

if len(history_df) == 0:
    print('Chua co history de ve. Hay chay cell train truoc.')
else:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.ravel()
    axes[0].plot(history_df['epoch'], history_df['train_accuracy'], marker='o')
    axes[0].set_title('Train accuracy')
    axes[0].set_xlabel('Epoch')

    axes[1].plot(history_df['epoch'], history_df['train_grad_loss'], marker='o', label='L_grad')
    axes[1].plot(history_df['epoch'], history_df['train_loss'], marker='o', label='L_TACS')
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    axes[2].plot(history_df['epoch'], history_df['train_policy_loss'], marker='o')
    axes[2].set_title('Policy loss')
    axes[2].set_xlabel('Epoch')

    axes[3].plot(history_df['epoch'], history_df['reward_mean'], marker='o')
    axes[3].fill_between(
        history_df['epoch'],
        history_df['reward_mean'] - history_df['reward_std'],
        history_df['reward_mean'] + history_df['reward_std'],
        alpha=0.2,
    )
    axes[3].axhline(0, color='black', linewidth=1)
    axes[3].set_title('Reward mean +/- std')
    axes[3].set_xlabel('Epoch')

    axes[4].plot(history_df['epoch'], history_df['policy_entropy'], marker='o')
    axes[4].set_title('Policy entropy')
    axes[4].set_xlabel('Epoch')

    axes[5].plot(history_df['epoch'], history_df['grad_selected_same_class_rate'], marker='o', label='Gumbel path')
    axes[5].plot(history_df['epoch'], history_df['policy_selected_same_class_rate'], marker='o', label='Policy sample')
    axes[5].set_title('Selected context same-class rate')
    axes[5].set_xlabel('Epoch')
    axes[5].legend()

    for ax in axes:
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    display(history_df.tail())


## 7. Official test accuracy

Cell này load final checkpoint; nếu final chưa có vì training bị dừng giữa chừng thì dùng `checkpoint_last.pt`. Khi test, Selector score toàn bộ candidate pool và chọn context có utility cao nhất bằng `argmax`, sau đó downstream classifier dự đoán class. Metric chính là `test_accuracy`.


In [ ]:
def macro_f1_score_np(y_true, y_pred, num_classes):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    f1_values = []
    for class_id in range(num_classes):
        tp = np.sum((y_true == class_id) & (y_pred == class_id))
        fp = np.sum((y_true != class_id) & (y_pred == class_id))
        fn = np.sum((y_true == class_id) & (y_pred != class_id))
        denom = 2 * tp + fp + fn
        f1_values.append(0.0 if denom == 0 else (2 * tp) / denom)
    return float(np.mean(f1_values))


def open_image_from_row(row, transform, crop_bbox=False):
    image = Image.open(row['image_path']).convert('RGB')
    if crop_bbox:
        left = max(0, int(row['bbox_x1']) - 1)
        top = max(0, int(row['bbox_y1']) - 1)
        right = int(row['bbox_x2'])
        bottom = int(row['bbox_y2'])
        image = image.crop((left, top, right, bottom))
    return transform(image)


def load_context_batch(candidate_annotations, context_positions, transform=eval_transform, crop_bbox=USE_BBOX_CROP):
    images = []
    for position in context_positions:
        row = candidate_annotations.iloc[int(position)]
        images.append(open_image_from_row(row, transform, crop_bbox=crop_bbox))
    return torch.stack(images, dim=0)


def extract_selector_embeddings(selector_model, annotations, desc='selector embeddings'):
    selector_model.eval()
    dataset = SimpleImageDataset(annotations, eval_transform, crop_bbox=USE_BBOX_CROP)
    loader = DataLoader(dataset, batch_size=SELECTOR_EMBED_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    embeddings = []
    with torch.inference_mode():
        for images, *_ in tqdm(loader, desc=desc, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                features = selector_model.encode_images(images)
            embeddings.append(features.float().cpu())
    return torch.cat(embeddings, dim=0)


def evaluate_tacs_model(selector_model, task_network, loader, top_k=TOP_K):
    selector_model.eval()
    task_network.eval()
    candidate_embeddings = extract_selector_embeddings(selector_model, candidate_df, desc='candidate selector embeddings')
    candidate_embeddings_device = candidate_embeddings.to(DEVICE)
    total_loss = 0.0
    all_targets = []
    all_predictions = []
    selected_same_class = []
    selected_scores = []
    rows = []
    pred_top_k = min(top_k, NUM_CLASSES)

    with torch.inference_mode():
        for query_images, query_indices, file_names, labels in tqdm(loader, desc='test', leave=False):
            query_images = query_images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                query_embeddings = selector_model.encode_images(query_images)
            selector_scores = query_embeddings.float() @ candidate_embeddings_device.T
            best_scores, context_positions = selector_scores.max(dim=1)
            context_images = load_context_batch(candidate_df, context_positions.detach().cpu().tolist()).to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                logits = task_network(query_images, context_images)
                loss = criterion(logits, labels)
            probs = torch.softmax(logits, dim=1).cpu()
            values, indices = torch.topk(probs, k=pred_top_k, dim=1)
            predictions = logits.argmax(dim=1).detach().cpu()
            labels_cpu = labels.detach().cpu()
            context_positions_cpu = context_positions.detach().cpu().tolist()
            best_scores_cpu = best_scores.detach().cpu().tolist()
            total_loss += loss.item() * query_images.size(0)
            all_targets.extend(labels_cpu.tolist())
            all_predictions.extend(predictions.tolist())

            for fname, label, pred, context_pos, selector_score, value_row, index_row in zip(file_names, labels_cpu, predictions, context_positions_cpu, best_scores_cpu, values, indices):
                label_id = int(label)
                pred_id = int(pred)
                context_row = candidate_df.iloc[int(context_pos)]
                context_label_id = int(context_row['class_id'])
                top_class_ids = [int(index) for index in index_row]
                same_class = bool(label_id == context_label_id)
                selected_same_class.append(same_class)
                selected_scores.append(float(selector_score))
                rows.append({
                    'fname': fname,
                    'true_class_id': label_id,
                    'true_car_name': class_names[label_id],
                    'selected_context_fname': context_row['fname'],
                    'selected_context_class_id': context_label_id,
                    'selected_context_car_name': class_names[context_label_id],
                    'selected_context_pool_position': int(context_pos),
                    'selected_context_train_row_id': int(context_row['row_id']),
                    'selector_score': float(selector_score),
                    'selected_context_same_class': same_class,
                    'predicted_class_id': pred_id,
                    'predicted_car_name': class_names[pred_id],
                    'correct': bool(pred_id == label_id),
                    'confidence': float(value_row[0]),
                    'top_k_class_ids': '|'.join(str(index) for index in top_class_ids),
                    'top_k_car_names': '|'.join(class_names[index] for index in top_class_ids),
                    'top_k_probabilities': '|'.join(f'{float(value):.6f}' for value in value_row),
                })

    y_true = np.asarray(all_targets)
    y_pred = np.asarray(all_predictions)
    return {
        'test_loss': total_loss / len(loader.dataset),
        'test_accuracy': float(np.mean(y_true == y_pred)),
        'test_macro_f1': macro_f1_score_np(y_true, y_pred, NUM_CLASSES),
        'selected_context_same_class_rate': float(np.mean(selected_same_class)) if selected_same_class else 0.0,
        'mean_selector_score': float(np.mean(selected_scores)) if selected_scores else 0.0,
        'predictions': pd.DataFrame(rows),
        'candidate_selector_embeddings': candidate_embeddings,
    }


evaluation_checkpoint_path = checkpoint_path if checkpoint_path.is_file() else last_checkpoint_path
saved_checkpoint = torch.load(evaluation_checkpoint_path, map_location=DEVICE)
selector.load_state_dict(saved_checkpoint['selector_state_dict'])
task_model.load_state_dict(saved_checkpoint['task_model_state_dict'])
selector.eval()
task_model.eval()
print('Loaded checkpoint for evaluation:', evaluation_checkpoint_path.resolve())

results = evaluate_tacs_model(selector, task_model, test_loader, top_k=TOP_K)
candidate_selector_embeddings = results.pop('candidate_selector_embeddings')
test_prediction_df = results.pop('predictions')
test_prediction_df.to_csv(OUT_PUT_DIR / 'test_predictions.csv', index=False)
test_prediction_df[[
    'fname',
    'true_class_id',
    'true_car_name',
    'selected_context_fname',
    'selected_context_class_id',
    'selected_context_car_name',
    'selected_context_pool_position',
    'selected_context_train_row_id',
    'selector_score',
    'selected_context_same_class',
]].to_csv(OUT_PUT_DIR / 'test_tacs_context_pairs.csv', index=False)
torch.save({'candidate_selector_embeddings': candidate_selector_embeddings, 'candidate_pool_positions': torch.as_tensor(candidate_pool_positions, dtype=torch.long)}, OUT_PUT_DIR / 'tacs_selector_candidate_embeddings.pt')
metrics = {
    'method': 'TACS',
    'test_accuracy': results['test_accuracy'],
    'test_loss': results['test_loss'],
    'test_macro_f1': results['test_macro_f1'],
    'selected_context_same_class_rate': results['selected_context_same_class_rate'],
    'mean_selector_score': results['mean_selector_score'],
    'num_classes': NUM_CLASSES,
    'standard_train_size': len(train_dataset),
    'standard_test_size': len(test_dataset),
    'architecture': BACKBONE_NAME,
    'pretraining': 'DINOv3',
    'fusion_mode': getattr(task_model, 'active_fusion_mode', saved_checkpoint.get('fusion_mode', 'unknown')),
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'seed': SEED,
    'use_bbox_crop': USE_BBOX_CROP,
    'candidate_pool_fraction': CANDIDATE_POOL_FRACTION,
    'candidate_pool_size': len(candidate_df),
    'selector_candidates_per_query': SELECTOR_CANDIDATES_PER_QUERY,
    'gumbel_temperature': GUMBEL_TEMPERATURE,
    'hybrid_lambda': HYBRID_LAMBDA,
    'policy_entropy_weight': POLICY_ENTROPY_WEIGHT,
    'selector_normalize_features': SELECTOR_NORMALIZE_FEATURES,
    'checkpoint': str(evaluation_checkpoint_path),
}
with open(OUT_PUT_DIR / 'metrics.json', 'w', encoding='utf-8') as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)
print(f"test_accuracy: {metrics['test_accuracy']:.4f} ({metrics['test_accuracy'] * 100:.2f}%)")
print(f"test_loss: {metrics['test_loss']:.4f}")
print(f"test_macro_f1: {metrics['test_macro_f1']:.4f}")
print(f"selected_context_same_class_rate: {metrics['selected_context_same_class_rate']:.4f}")
print('Saved test predictions:', (OUT_PUT_DIR / 'test_predictions.csv').resolve())
print('Saved TACS context pairs:', (OUT_PUT_DIR / 'test_tacs_context_pairs.csv').resolve())
print('Saved metrics:', (OUT_PUT_DIR / 'metrics.json').resolve())
display(test_prediction_df.head())


## 8. Inference và load checkpoint

Dùng các hàm này để test nhanh một ảnh đơn. Selector sẽ chọn context từ candidate pool đã train, rồi task model dự đoán class.


In [ ]:
def ensure_candidate_selector_embeddings(selector_model=selector):
    global candidate_selector_embeddings
    if 'candidate_selector_embeddings' not in globals():
        cache_path = OUT_PUT_DIR / 'tacs_selector_candidate_embeddings.pt'
        if cache_path.is_file():
            cached = torch.load(cache_path, map_location='cpu')
            candidate_selector_embeddings = cached['candidate_selector_embeddings']
        else:
            candidate_selector_embeddings = extract_selector_embeddings(selector_model, candidate_df, desc='candidate selector embeddings')
    return candidate_selector_embeddings


def select_tacs_context_for_image(image_path, selector_model=selector):
    image_path = Path(image_path)
    selector_model.eval()
    candidate_embeddings = ensure_candidate_selector_embeddings(selector_model).to(DEVICE)
    image = eval_transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            query_embedding = selector_model.encode_images(image)
        scores = query_embedding.float() @ candidate_embeddings.T
        best_score, best_position = scores.max(dim=1)
    context_row = candidate_df.iloc[int(best_position.item())]
    return context_row, float(best_score.item())


def predict_one_image_with_tacs(image_path, selector_model=selector, network=task_model, prediction_top_k=TOP_K):
    image_path = Path(image_path)
    context_row, selector_score = select_tacs_context_for_image(image_path, selector_model=selector_model)
    context_image_path = Path(context_row['image_path'])
    query_image = eval_transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    context_image = eval_transform(Image.open(context_image_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    network.eval()
    with torch.inference_mode():
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            logits = network(query_image, context_image)
        probs = torch.softmax(logits, dim=1)[0]
        values, indices = torch.topk(probs, k=min(prediction_top_k, NUM_CLASSES))
    result = pd.DataFrame({
        'rank': np.arange(1, len(indices) + 1),
        'class_id': [int(index) for index in indices.cpu()],
        'car_name': [class_names[int(index)] for index in indices.cpu()],
        'probability': [float(value) for value in values.cpu()],
    })
    print('Query image:', image_path)
    print('Selected context image:', context_image_path)
    print('Selected context label:', class_names[int(context_row['class_id'])])
    print(f'Selector score: {selector_score:.4f}')
    return result


def load_tacs_model_for_inference(checkpoint_file, device=DEVICE):
    saved = torch.load(checkpoint_file, map_location=device)
    loaded_selector = TACSSelector(normalize_features=saved.get('selector_normalize_features', SELECTOR_NORMALIZE_FEATURES))
    loaded_task_model = DinoV3ContextClassifier(
        num_classes=len(saved['class_names']),
        feature_dim=saved.get('feature_dim', FEATURE_DIM),
    )
    loaded_selector.load_state_dict(saved['selector_state_dict'])
    loaded_task_model.load_state_dict(saved['task_model_state_dict'])
    loaded_selector.to(device).eval()
    loaded_task_model.to(device).eval()
    return loaded_selector, loaded_task_model, saved['class_names']

print('Method:', 'TACS')
print('Final checkpoint:', checkpoint_path.resolve())
print('Latest checkpoint:', last_checkpoint_path.resolve())


## 9. Ghi chú benchmark

- Method này tương ứng hàng **TACS (Ours)** trong Table 2.
- Candidate pool lấy từ 20% train split và dùng cho cả train/test.
- Train dùng `SELECTOR_CANDIDATES_PER_QUERY` candidate/query để vừa sức Colab; test dùng full candidate pool.
- Selector học utility theo task reward, không chỉ similarity DINO frozen.
- Downstream classifier nhận pair `(query, selected_context)`.
- Không dùng validation split và không dùng test để chọn epoch.
- Metric chính cho Cars là `test_accuracy`.
- Những file nên giữ để so sánh method: `metrics.json`, `training_history.csv`, `test_predictions.csv`, `candidate_pool.csv`, `test_tacs_context_pairs.csv`, `tacs_selector_candidate_embeddings.pt`, `checkpoint_last.pt`, và final checkpoint.
